# J4-A — Des données propres à un adaptateur LoRA

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 4 · 160 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Auditer un petit jeu d’instructions, son format et ses partitions.
- Expliquer quels poids LoRA apprend et ce que QLoRA quantifie.
- Comparer le modèle initial et adapté sur les mêmes requêtes sans inventer de gain.

## Parcours de l’après-midi

| Étape | Minutes |
| Curation et contrat de données | 35 |
| Format chat, loss masquée et LoRA | 35 |
| Entraînement borné et suivi | 40 |
| Comparaison, limites et export | 50 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'datasets': '4.1.1', 'peft': '0.17.1', 'trl': '0.23.1', 'bitsandbytes': '0.47.0'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
versions["torch"] = torch.__version__
results["device"] = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print("Périphérique :", results["device"])
if DEVICE == "cuda":
    print("VRAM totale (Gio) :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print("Pas de GPU : les sections indiquées restent utilisables ; ne pas présenter un modèle non entraîné comme adapté.")


## 1. Le contrat avant le GPU

Notre assistant doit répondre en français, indiquer une prochaine action et ne pas inventer
de date, remboursement ou accès à un dossier. Les réponses ci-dessous constituent une politique
**fictive** de support, pas les conditions commerciales d’une entreprise. Le SFT apprend à imiter
les réponses fournies : il ne crée pas une connexion aux commandes et ne garantit pas leur vérité.

Le corpus reprend les scénarios des tickets, avec des réponses originales. Les deux paraphrases
d’un scénario partagent la réponse et restent ensemble. À cette échelle, le risque de mémorisation
est élevé ; la séance sert à observer le pipeline et ses limites.


In [ ]:
SUPPORT = {'livraison': [('Le suivi de mon colis ne bouge plus depuis lundi.', 'Mon paquet semble bloqué au centre de tri.'), ('Le livreur indique livré mais ma boîte est vide.', "Je n'ai rien reçu alors que le suivi annonce une remise."), ('Puis-je choisir un autre point relais ?', 'Je voudrais faire déposer ma commande dans un relais différent.'), ("L'adresse sur mon envoi est incomplète.", 'Il manque le numéro de rue pour la livraison.'), ('Le paquet est reparti chez vous avant mon passage.', "Je n'ai pas pu récupérer le colis à temps au relais."), ('Les deux articles voyagent-ils dans le même carton ?', "Je n'ai qu'un numéro de suivi pour plusieurs produits."), ('Le transporteur a reporté son passage sans date.', "Aucune nouvelle tentative de livraison n'est annoncée."), ('Mon colis doit arriver au bureau fermé samedi.', 'La réception de mon entreprise sera absente lors du passage.')], 'facturation': [("J'ai été débité deux fois pour la même commande.", 'Deux paiements identiques apparaissent sur mon relevé.'), ('Où récupérer la facture de mon achat ?', "J'ai besoin d'un justificatif comptable pour la commande."), ("La remise affichée n'a pas été appliquée au paiement.", 'Le total payé ignore le code promotionnel saisi.'), ('Le paiement est refusé alors que ma carte fonctionne ailleurs.', "Votre page n'accepte pas ma carte bancaire."), ('Le nom de mon entreprise manque sur la facture.', 'Pouvez-vous corriger les informations de facturation ?'), ("Une somme reste en attente après l'annulation.", "L'autorisation bancaire semble encore bloquée."), ('Le reçu est dans une devise différente de mon panier.', 'Je ne comprends pas la conversion sur mon justificatif.'), ('La taxe indiquée sur le document paraît incorrecte.', 'Le montant de TVA de ma facture mérite une vérification.')], 'compte': [("J'ai oublié mon mot de passe.", 'Je ne me souviens plus du code pour accéder à mon espace.'), ("Le courriel de réinitialisation n'arrive pas.", "Le lien pour retrouver mon accès n'est pas dans ma boîte mail."), ("Je voudrais changer l'adresse email de mon profil.", 'Mon espace utilise encore mon ancien courriel.'), ('Comment fermer définitivement mon compte ?', 'Je souhaite supprimer mon profil client.'), ('Le code de connexion à deux facteurs est expiré.', 'La double authentification bloque mon accès.'), ('Je ne reconnais pas une connexion récente.', 'Mon historique montre un accès qui ne vient pas de moi.'), ('Je veux récupérer les données de mon profil.', 'Comment obtenir une copie des informations de mon compte ?'), ('Mon adresse mail semble associée à deux profils.', 'Je retrouve deux espaces clients pour la même identité.')], 'retour': [("La taille reçue ne me convient pas, je veux renvoyer l'article.", 'Je souhaite retourner ce vêtement trop petit.'), ('Le produit arrivé cassé doit vous être retourné.', 'Je voudrais organiser le renvoi de cet article abîmé.'), ("Où trouver l'étiquette de retour ?", "Je n'arrive pas à télécharger le bordereau de renvoi."), ('Mon retour a été reçu, où en est son traitement ?', "Avez-vous vérifié le produit que j'ai renvoyé ?"), ("Je préfère échanger l'article plutôt que le garder.", 'Est-il possible de remplacer ce produit par une autre taille ?'), ("J'ai jeté l'emballage, puis-je tout de même renvoyer ?", "Le carton d'origine est perdu pour mon retour."), ("Je souhaite renvoyer seulement une partie de l'achat.", "Comment retourner un seul article d'une commande multiple ?"), ('Le transporteur refuse mon paquet de retour trop volumineux.', 'Le renvoi ne rentre pas dans le format accepté au relais.')], 'technique': [("L'application se ferme quand j'ouvre mon panier.", 'Votre appli plante au moment de voir les articles ajoutés.'), ('La page produit reste blanche sur mon navigateur.', 'Les fiches articles ne se chargent plus sur le site.'), ('Le bouton de recherche ne fait rien.', 'Je saisis un mot mais la recherche du site ne démarre pas.'), ("L'image du produit ne s'affiche pas.", 'Toutes les photos sont remplacées par une icône cassée.'), ('La version mobile masque le bouton continuer.', "Sur mon téléphone je ne peux pas passer à l'étape suivante."), ("Le site boucle lorsqu'on accepte les cookies.", 'La fenêtre de consentement réapparaît sans cesse.'), ('Votre formulaire refuse les caractères accentués.', "Le champ de saisie signale une erreur quand j'écris é."), ("La page est inutilisable avec mon lecteur d'écran.", "La navigation au clavier n'atteint pas les boutons du site.")]}
REPLIES = {'livraison': ["Je comprends l'attente. Consultez la dernière étape du suivi. Si elle reste inchangée, transmettez la référence de commande au support. Je ne peux pas confirmer une date de livraison.", "Vérifiez le lieu de dépôt indiqué et demandez aux personnes susceptibles d'avoir réceptionné le colis. Si vous ne le retrouvez pas, contactez le support avec la référence de commande.", "Consultez les options de redirection proposées dans le suivi du transporteur. Si aucune option n'apparaît, contactez le support. Un changement de relais n'est pas garanti.", "Contactez rapidement le support avec la référence de commande et l'adresse corrigée. La modification dépend de l'état d'expédition ; je ne peux pas la confirmer ici.", "Le suivi permet de vérifier le retour à l'expéditeur. Contactez ensuite le support avec la référence de commande pour examiner les options, sans promettre de réexpédition automatique.", 'Consultez les détails de commande pour rechercher plusieurs expéditions. Le support peut vérifier les références de suivi des différents articles.', 'Consultez les dernières informations du transporteur puis contactez le support avec la référence du suivi. Aucune nouvelle date ne peut être confirmée ici.', 'Vérifiez si le transporteur propose un report ou une autre remise. Prévenez le support si le destinataire ne sera pas disponible ; la modification reste à confirmer.'], 'facturation': ["Vérifiez si les deux lignes sont des débits effectifs ou si l'une est une autorisation en attente. Contactez le support avec la référence de commande et un relevé masqué. Ne communiquez jamais le numéro complet de votre carte.", "Ouvrez votre espace client, puis la commande concernée et la rubrique facture. Si le document n'est pas disponible, contactez le support avec la référence de commande.", "Vérifiez les conditions du code et le récapitulatif de commande. Contactez le support avec le code et la référence de commande ; aucun remboursement n'est confirmé ici.", "Vérifiez les informations saisies et contactez votre banque si le refus persiste. N'envoyez ni numéro complet de carte ni cryptogramme au support.", 'Transmettez au support la référence de facture et les informations à corriger. Le service comptable doit confirmer si une correction est possible.', "Une autorisation en attente n'est pas forcément un débit définitif. Vérifiez le statut auprès de votre banque et contactez le support avec la référence de commande si nécessaire.", 'Comparez la devise de la commande à celle du relevé. Votre banque peut appliquer une conversion. Contactez le support avec un justificatif masqué pour vérifier le montant facturé.', 'Transmettez la référence de facture et le montant contesté au support pour vérification comptable. Je ne peux pas valider un calcul de taxe sans le document.'], 'compte': ['Utilisez le lien de réinitialisation du mot de passe sur la page de connexion. Choisissez un mot de passe unique et ne communiquez aucun code de sécurité au support.', "Vérifiez les courriers indésirables et l'adresse saisie, puis demandez un nouveau lien. Si le problème persiste, contactez le support sans transmettre de mot de passe.", "Consultez les paramètres de votre profil pour modifier le courriel et suivre la vérification demandée. Si vous n'avez plus accès à l'ancienne adresse, contactez le support.", 'Consultez la rubrique de gestion du compte ou contactez le support pour demander sa suppression. Le support vous indiquera la procédure de vérification et les conséquences.', "Demandez un nouveau code et vérifiez l'heure de votre appareil. Utilisez la procédure officielle de récupération si nécessaire. Ne partagez pas vos codes à usage unique.", 'Changez votre mot de passe depuis le site officiel et vérifiez les sessions actives. Contactez le support pour signaler cet accès sans communiquer vos codes de sécurité.', "Utilisez la rubrique de confidentialité du compte ou contactez le support pour demander une copie de vos données. Une vérification d'identité peut être nécessaire via la procédure officielle.", 'Contactez le support pour vérifier les profils concernés. Ne créez pas un nouveau compte avant cette vérification et ne transmettez pas de mot de passe.'], 'retour': ["Consultez les conditions de retour de votre commande puis ouvrez une demande depuis votre espace client. L'éligibilité et les frais éventuels doivent être vérifiés avant l'envoi.", "Prenez des photos du dommage et contactez le support avec la référence de commande. Attendez les instructions de retour avant d'expédier l'article.", 'Ouvrez la demande de retour dans votre espace client pour chercher le bordereau. Si le téléchargement échoue, contactez le support avec la référence du retour.', 'Consultez le statut de votre demande puis contactez le support avec le numéro de retour et le suivi. Je ne peux pas confirmer le remboursement ni son délai ici.', "Consultez les conditions d'échange et les tailles disponibles, puis contactez le support ou ouvrez une demande. La disponibilité du remplacement doit être confirmée.", "Consultez les conditions de retour puis demandez au support si un autre emballage est accepté. Protégez le produit et attendez la confirmation avant l'envoi.", "Sélectionnez l'article concerné dans la demande de retour. Vérifiez les instructions de séparation des colis et les conditions applicables auprès du support.", "Contactez le support avec les dimensions du colis et la référence du retour pour connaître une solution de transport adaptée. N'avancez pas de frais sans confirmation."], 'technique': ["Mettez l'application à jour et relancez-la. Si le problème persiste, transmettez au support le modèle de téléphone, la version de l'application et les étapes du plantage.", "Actualisez la page et essayez un autre navigateur. Signalez au support la page concernée, le navigateur et le moment de l'erreur si elle persiste.", 'Essayez une nouvelle requête puis un autre navigateur. Si le bouton ne répond toujours pas, transmettez les étapes et la version du navigateur au support.', 'Vérifiez votre connexion et actualisez la page. Si les images restent absentes, indiquez au support la page, le navigateur et une capture sans données personnelles.', "Essayez d'actualiser la page et notez le navigateur ainsi que la taille de l'écran. Signalez au support le bouton masqué ; évitez de communiquer des informations de paiement dans une capture.", 'Testez un autre navigateur et notez les choix de consentement qui déclenchent la boucle. Transmettez ces étapes au support technique.', "Signalez le champ concerné, le navigateur et un exemple fictif contenant l'accent au support technique. Ne remplacez pas durablement votre identité par une valeur incorrecte.", "Signalez au support les boutons inaccessibles, le lecteur d'écran et le navigateur utilisés. Demandez une solution d'accès adaptée pour poursuivre votre démarche."]}
LABELS = sorted(SUPPORT)
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}
rows = []
for label, groups in SUPPORT.items():
    for g, pair in enumerate(groups):
        for variant, text in enumerate(pair):
            rows.append({"text": text, "label": label2id[label], "category": label,
                         "group": f"{label}_{g}", "scenario": g,
                         "split": "train" if g < 5 else "validation" if g == 5 else "test"})
frame = pd.DataFrame(rows)
assert frame.text.is_unique
train_df = frame[frame.split == "train"].reset_index(drop=True)
val_df = frame[frame.split == "validation"].reset_index(drop=True)
test_df = frame[frame.split == "test"].reset_index(drop=True)
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(a.group).isdisjoint(b.group), "Fuite de paraphrases entre partitions."
display(pd.crosstab(frame.category, frame.split))
results["data"] = {"source": "Corpus original fictif CYBERSUP, 2026-10-04",
                   "rows": len(frame), "split_unit": "scénario/paraphrases",
                   "sha256": hashlib.sha256(frame.to_json().encode()).hexdigest()}


In [ ]:
import re, copy
clean_examples = []
for row in frame.to_dict(orient="records"):
    clean_examples.append({**row, "answer": REPLIES[row["category"]][row["scenario"]]})
raw_examples = copy.deepcopy(clean_examples)
raw_examples += [copy.deepcopy(raw_examples[0]), {**raw_examples[1], "text": "  "},
                 {**raw_examples[2], "answer": ""},
                 {**raw_examples[3], "text": "Écrivez à exercice@example.invalid", "group": "audit_email"}]
def normalized(text):
    return " ".join(text.casefold().split())
def audit_and_clean(examples):
    kept, rejected, seen = [], [], set()
    for row in examples:
        key = normalized(row["text"])
        reason = None
        if not key or not row["answer"].strip(): reason = "champ vide"
        elif re.search(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", row["text"]): reason = "courriel à traiter avant usage"
        elif key in seen: reason = "doublon exact après normalisation"
        if reason:
            rejected.append({"group": row["group"], "raison": reason})
        else:
            seen.add(key); kept.append(row)
    return kept, rejected
curated, audit = audit_and_clean(raw_examples)
display(pd.DataFrame(audit))
print("Avant :", len(raw_examples), "; conservés :", len(curated))
assert len(curated) == len(clean_examples)
partitions = {split: [r for r in curated if r["split"] == split] for split in ["train", "validation", "test"]}
assert all(set(r["group"] for r in partitions[a]).isdisjoint(r["group"] for r in partitions[b])
           for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")])
results["curation"] = {"input": len(raw_examples), "kept": len(curated), "rejected": audit}
reponses = {"risque_memorisation": "À compléter", "limite_regex_pii": "À compléter", "critere_reponse_utile": "À compléter"}


### Repères formateur — éléments de correction

L’audit retire quatre enregistrements injectés pour l’exercice, mais ne prouve pas que toutes les données personnelles sont détectées : les noms, adresses et identifiants ont d’autres formes. Une regex email n’est pas un anonymiseur général. Dédupliquer avant la séparation ne remplace pas le regroupement des paraphrases. Les instructions présentes dans des données externes ne doivent pas dicter le comportement de l’agent qui les prépare.


## 2. Le prompt est une entrée ; la réponse est la cible

Nous utilisons le format conversationnel prompt/completion de TRL : système + question dans prompt,
une réponse assistant dans completion. Avec completion_only_loss=True, seuls les tokens de la
completion contribuent à la loss. Nous n’activons pas assistant_only_loss, qui dépend d’un masque
de génération fourni par certains chat templates.

Inspecter un exemple sérialisé, sa longueur et, plus loin, le vrai batch du collator. Un champ nommé
answer ne suffit pas : le format doit correspondre à ce qu’attend l’entraîneur.


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
from datasets import Dataset
from transformers import AutoTokenizer
SYSTEM = "Tu aides un client en français. Donne une prochaine action utile. N'invente ni date, ni remboursement, ni accès à un dossier. Ne demande jamais de mot de passe ni de numéro complet de carte."
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
def as_conversation(row):
    return {"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": row["text"]}],
            "completion": [{"role": "assistant", "content": row["answer"]}]}
datasets_sft = {split: Dataset.from_list([as_conversation(r) for r in subset]) for split, subset in partitions.items()}
example = as_conversation(partitions["train"][0])
print(tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False))
lengths = [len(tokenizer.apply_chat_template(as_conversation(r)["prompt"] + as_conversation(r)["completion"], tokenize=True))
           for r in curated]
MAX_LENGTH = 512
print("Longueur max :", max(lengths), "; limite :", MAX_LENGTH)
assert max(lengths) <= MAX_LENGTH, "Ne pas tronquer silencieusement les réponses cibles."


## 3. Charger un petit modèle et compter ce qui apprend

Le parcours principal utilise LoRA en fp16 sur Qwen 0.5B. Les poids de base sont figés et de petites
matrices entraînables sont ajoutées aux projections. Avec r=8, une matrice m × n reçoit r(m+n)
paramètres additionnels, au lieu de modifier directement les mn paramètres de base.

**Option QLoRA.** Après avoir terminé LoRA, redémarrer le runtime, mettre USE_QLORA=True et garder
les mêmes données et le même nombre de pas. NF4 quantifie les poids de base, pas tous les tenseurs
de l’entraînement. Les calculs et les adaptateurs gardent une précision plus élevée. À 0.5B, le gain
de mémoire peut être moins spectaculaire que sur un grand modèle. Mesurer au lieu de promettre.

Sans GPU : poursuivre curation, format et calcul des paramètres. L’entraînement est explicitement
non exécuté. En cas de mémoire insuffisante : nouveau runtime, batch=1, puis MAX_LENGTH=256 seulement
si l’assertion de longueur passe. Ne pas réduire la qualité de l’évaluation pour masquer un problème.


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
USE_QLORA = False
RUN_TRAINING = DEVICE == "cuda"
RANK = 8
MAX_STEPS = 30  # Budget pédagogique borné ; augmenter après diagnostic, pas après lecture du test.
model = None
lora_config = LoraConfig(r=RANK, lora_alpha=16, lora_dropout=0.05,
                         target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM")
if RUN_TRAINING:
    set_seed(SEED)
    kwargs = {"torch_dtype": torch.float16}
    if USE_QLORA:
        kwargs.update(quantization_config=BitsAndBytesConfig(load_in_4bit=True,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16),
            device_map={"": 0})
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    if USE_QLORA:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.to(DEVICE)
    model = get_peft_model(model, lora_config)
    model.config.use_cache = False
    model.print_trainable_parameters()
    learned = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    assert 0 < learned < total
    results["parameters"] = {"trainable": learned, "stored_parameter_count": total,
                              "warning": "Le stockage quantifié ne se résume pas au nombre logique de paramètres."}
else:
    print("Pas d'entraînement. Calcul illustratif LoRA pour m=n=1024 :", RANK * (1024 + 1024), "paramètres.")


In [ ]:
# Même fonction, même prompt, même décodage pour la comparaison avant/après.
def answer_question(question, disable_adapter=False):
    from contextlib import nullcontext
    model.eval()
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_tensors="pt", return_dict=True).to(DEVICE)
    context = model.disable_adapter() if disable_adapter else nullcontext()
    with context, torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=96, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id, use_cache=True)
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
validation_probes = [r for r in partitions["validation"] if r["text"] == SUPPORT[r["category"]][5][0]]
before = []
if model is not None:
    before = [answer_question(r["text"], disable_adapter=True) for r in validation_probes]
    display(pd.DataFrame({"question": [r["text"] for r in validation_probes], "avant": before}))


## 4. Vérifier le masque réellement utilisé, puis entraîner

Lire les tokens appris et ignorés dans le batch. La preuve attendue est la présence de -100
sur le prompt et de labels supervisés sur la réponse. Les positions de padding sont également ignorées.
Observer loss train, loss validation, temps et mémoire ; une baisse de loss n’est pas une mesure
suffisante de factualité ni d’utilité.


In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = None
if model is not None:
    config = SFTConfig(output_dir="checkpoints_sft", max_steps=MAX_STEPS,
        per_device_train_batch_size=1, gradient_accumulation_steps=8, per_device_eval_batch_size=1,
        learning_rate=1e-4, warmup_ratio=0.05, max_length=MAX_LENGTH, packing=False,
        completion_only_loss=True, assistant_only_loss=False, eos_token="<|im_end|>",
        fp16=True, bf16=False, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="steps", eval_steps=10, save_strategy="no", logging_steps=5,
        report_to="none", seed=SEED, data_seed=SEED, push_to_hub=False,
        optim="adamw_torch", dataloader_num_workers=0)
    trainer = SFTTrainer(model=model, args=config, train_dataset=datasets_sft["train"],
                         eval_dataset=datasets_sft["validation"], processing_class=tokenizer)
    sample_batch = trainer.data_collator([trainer.train_dataset[0]])
    token_ids, supervised = sample_batch["input_ids"][0], sample_batch["labels"][0]
    assert (supervised == -100).any() and (supervised != -100).any()
    first_learned = int(torch.nonzero(supervised != -100)[0])
    assert first_learned > 0
    print("PROMPT IGNORÉ :", tokenizer.decode(token_ids[:first_learned]))
    print("CIBLE APPRISE :", tokenizer.decode(supervised[supervised != -100]))
    display(pd.DataFrame({"token": tokenizer.convert_ids_to_tokens(token_ids.tolist()),
                          "appris": (supervised != -100).tolist()}).tail(30))
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    trainer.train()
    results["sft"] = {"seconds": time.perf_counter() - start,
                       "peak_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
                       "qlora": USE_QLORA, "rank": RANK, "steps": MAX_STEPS,
                       "model": MODEL_ID, "max_length": MAX_LENGTH,
                       "validation": trainer.evaluate(), "history": trainer.state.log_history}
else:
    results["sft"] = None
    print("SFT non exécuté ; aucune mesure GPU ni qualité adaptée à déclarer.")


### Repères formateur — éléments de correction

Le premier label appris doit se trouver après le prompt, pas à l’index zéro. Vérifier visuellement que la question n’est pas apprise comme cible. On passe un modèle PEFT déjà construit à SFTTrainer sans ajouter à nouveau peft_config. use_cache est désactivé pour l’entraînement avec checkpointing, mais activé explicitement lors des petites générations. La VRAM allouée PyTorch ne couvre pas nécessairement toute la mémoire réservée par le processus.


## 5. Comparer en aveugle autant que possible

Pour chaque paire avant/après, attribuer 0, 1 ou 2 à trois critères : action utile, respect des limites
factuelles, respect du format demandé. Mélanger l’ordre A/B avant de noter ; faire annoter par l’autre
binôme, puis révéler la méthode. En cas de désaccord, conserver les deux avis et discuter la règle.

Le test final contient de nouveaux scénarios par rapport à train/validation. Après choix des réglages,
comparer une formulation par scénario de test, avec le même système et le même décodage.
Les résultats de cinq ou dix requêtes ne suffisent pas pour annoncer une amélioration générale.


In [ ]:
comparisons = []
if model is not None:
    after = [answer_question(r["text"]) for r in validation_probes]
    comparisons = [{"question": r["text"], "avant": a, "apres": z, "reference": r["answer"]}
                   for r, a, z in zip(validation_probes, before, after)]
    display(pd.DataFrame(comparisons))
final_test = []
if model is not None:
    # Ne relancer ce bloc qu'après avoir figé la configuration ; conserver le résultat de la première lecture.
    probes = [r for r in partitions["test"] if r["text"] == SUPPORT[r["category"]][r["scenario"]][0]]
    for row in probes:
        final_test.append({"group": row["group"], "question": row["text"],
            "avant": answer_question(row["text"], disable_adapter=True),
            "apres": answer_question(row["text"]), "reference": row["answer"]})
    display(pd.DataFrame(final_test))
human_ratings = []  # À remplir : group, méthode, action_0_2, factualite_0_2, format_0_2, justification.
results.update({"validation_pairs": comparisons, "test_pairs": final_test,
                "human_ratings": human_ratings, "reponses": reponses})
conclusion_binome = "À compléter : observations avant/après, coût, surapprentissage possible et prochaines données."


In [ ]:
import shutil
if model is not None:
    adapter_dir = Path("adaptateur_lora")
    model.save_pretrained(adapter_dir, safe_serialization=True)
    tokenizer.save_pretrained(adapter_dir)
    (adapter_dir / "provenance.json").write_text(json.dumps({"base_model": MODEL_ID, "versions": versions,
        "data": results["data"], "training": results["sft"], "train_groups": sorted(train_df.group.unique().tolist())},
        ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    (adapter_dir / "README.md").write_text(
        "---\nbase_model: " + MODEL_ID + "\nlibrary_name: peft\nlanguage: fr\nlicense: apache-2.0\n---\n"
        "# Adaptateur pédagogique CYBERSUP\n\nCorpus fictif français, usage pédagogique uniquement. "
        "Aucune connexion à des commandes réelles. Les poids du modèle de base doivent être chargés séparément. "
        "Pas de garantie de factualité ou de performance métier. Voir provenance.json pour le protocole.\n",
        encoding="utf-8")
    archive = shutil.make_archive("adaptateur_lora", "zip", root_dir=adapter_dir)
    print("Télécharger l'adaptateur avant de fermer Colab :", archive)
    assert (adapter_dir / "adapter_config.json").exists()
    assert (adapter_dir / "adapter_model.safetensors").exists()
else:
    print("Pas d'adaptateur à exporter.")


### Repères formateur — éléments de correction

LoRA apprend une correction à faible rang, pas une nouvelle base de faits accessible à la demande. QLoRA quantifie les poids gelés et apprend toujours des adaptateurs. Une absence de gain, voire une dégradation, est un résultat acceptable si le protocole est respecté. Sur ce corpus très court, 30 pas peuvent suffire à surapprendre ; davantage de pas ne corrige ni des réponses mauvaises ni une partition contaminée.


## Ressources pour prolonger

- [HF — SFT et fine-tuning](https://huggingface.co/learn/llm-course/en/chapter11/1)
- [TRL 0.23.1 — formats et loss de completion](https://huggingface.co/docs/trl/v0.23.1/en/sft_trainer)
- [PEFT — quantification et préparation](https://huggingface.co/docs/peft/en/developer_guides/quantization)
- [Modèle Qwen et licence](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)

Liens et API consultés le 4 octobre 2026. Les exercices et données de ce notebook sont originaux. Les téléchargements de modèles et l’exécution Colab doivent être vérifiés avant la séance.


## Trace à rendre

Compléter vos réponses et ajouter une observation appuyée par une sortie réelle.
Exécuter la dernière cellule, télécharger le JSON dans le panneau Fichiers de Colab,
puis télécharger votre notebook avec ses sorties. Les fichiers du runtime sont temporaires.
Ne pas remplacer une mesure absente par un chiffre plausible. Indiquer les étapes non exécutées.


In [ ]:
results["notebook"] = '05_j4_sft_lora'
results["conclusion_binome"] = globals().get("conclusion_binome", "À compléter")

result_file = OUTPUT / "j4_sft_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
